Link github:https://github.com/24110056-cmd/Tri_Tue_Nhan_Tao/tree/main/Model-Based%20Reflex%20Agent

MÔ HÌNH PEAS
- P (Performance): Làm sạch hết bụi trong thời gian ngắn nhất, tối ưu hóa quãng đường di chuyển (không đi lặp lại ô đã sạch), ít tốn điện.
- E (Environment): Ma trận sàn nhà m x n (0: sạch, 1: bẩn); vật cản, tường.
- A (Actuators): Bánh xe (Lên, Xuống, Trái, Phải), động cơ hút bụi.
- S (Sensors): Cảm biến vị trí hiện tại (x, y), cảm biến bụi tại ô đang đứng, Bộ nhớ (Internal State) lưu trữ bản đồ những ô đã đi qua.

PERCEPT:
- Vị trí hiện tại: (x, y).
- Trạng thái ô: Sạch (0) hoặc Bẩn (1).
- Model nội bộ: Ma trận đánh dấu những ô đã ghé thăm để ưu tiên các ô chưa đến.

RULES:
Tập luật của Model-based sẽ phức tạp hơn vì nó phải kiểm tra cả trạng thái thực tế và bộ nhớ:
- Nếu ô hiện tại có bụi thì HÚT BỤI.
- Nếu ô hiện tại sạch:
    + Kiểm tra các ô liền kề (Lên, Xuống, Trái, Phải).
    + Lọc ra danh sách các ô mà Model báo là "Chưa đến".
    + Nếu có ô chưa đến: Chọn ngẫu nhiên một ô trong danh sách đó.
    + Nếu tất cả ô liền kề đã đến: Chọn ngẫu nhiên một hướng hợp lệ để di chuyển tiếp.

In [1]:
import random
import time
class ModelBasedRobot:
    def __init__(self, m, n):
        self.m = m
        self.n = n
        self.grid = [[random.choice([0, 1]) for _ in range(n)] for _ in range(m)]
        self.x = random.randint(0, m - 1)
        self.y = random.randint(0, n - 1)
        self.model = [[0 for _ in range(n)] for _ in range(m)]
        self.step_count = 0
    def is_clean(self):
        return all(1 not in row for row in self.grid)
    def show_grid(self, step):
        print(f"BƯỚC {step}:")
        for i in range(self.m):
            row_display = []
            for j in range(self.n):
                if i == self.x and j == self.y:
                    row_display.append("R")
                else:
                    row_display.append(str(self.grid[i][j]))
            print(" ".join(row_display))
        print("-" * 25)

    def move(self):
        self.model[self.x][self.y] = 1
        if self.grid[self.x][self.y] == 1:
            print(f"-> [SENSORS]: Phát hiện bụi tại ({self.x}, {self.y}). Đang hút...")
            self.grid[self.x][self.y] = 0
            return "SUCK"
        directions = {
            "LÊN": (self.x - 1, self.y),
            "XUỐNG": (self.x + 1, self.y),
            "TRÁI": (self.x, self.y - 1),
            "PHẢI": (self.x, self.y + 1)
        }
        valid_neighbors = {k: v for k, v in directions.items() 
                          if 0 <= v[0] < self.m and 0 <= v[1] < self.n}
        unvisited = {k: v for k, v in valid_neighbors.items() 
                     if self.model[v[0]][v[1]] == 0}
        if unvisited:
            unvisited_list = [f"{k}{v}" for k, v in unvisited.items()]
            print(f"-> [MODEL]: Các ô liền kề chưa đến: {unvisited_list}")
            action = random.choice(list(unvisited.keys()))
            target = unvisited[action]
            print(f"==> Ưu tiên chọn ô chưa đến: {action} ({target[0]}, {target[1]})")
        else:
            action = random.choice(list(valid_neighbors.keys()))
            target = valid_neighbors[action]
            print(f"-> [MODEL]: Các ô liền kề đã ghé thăm hết.")
            print(f"==> Robot đi ngẫu nhiên để tìm vùng mới: {action} ({target[0]}, {target[1]})")
        self.x, self.y = target
        return action
if __name__ == "__main__":
    M, N = 3, 3
    robot = ModelBasedRobot(M, N)
    print(f"Khởi chạy Robot (Model-Based) trên lưới {M}x{N}")
    print("Ghi chú: R=Robot, 1=Bụi, 0=Sạch\n")
    step = 0
    while not robot.is_clean():
        step += 1
        robot.show_grid(step)
        robot.move()
        time.sleep(0.5)
    robot.show_grid(step + 1)
    print(f"HOÀN THÀNH! Sàn sạch sau {step} bước.")

Khởi chạy Robot (Model-Based) trên lưới 3x3
Ghi chú: R=Robot, 1=Bụi, 0=Sạch

BƯỚC 1:
1 1 1
1 1 1
0 R 1
-------------------------
-> [SENSORS]: Phát hiện bụi tại (2, 1). Đang hút...
BƯỚC 2:
1 1 1
1 1 1
0 R 1
-------------------------
-> [MODEL]: Các ô liền kề chưa đến: ['LÊN(1, 1)', 'TRÁI(2, 0)', 'PHẢI(2, 2)']
==> Ưu tiên chọn ô chưa đến: TRÁI (2, 0)
BƯỚC 3:
1 1 1
1 1 1
R 0 1
-------------------------
-> [MODEL]: Các ô liền kề chưa đến: ['LÊN(1, 0)']
==> Ưu tiên chọn ô chưa đến: LÊN (1, 0)
BƯỚC 4:
1 1 1
R 1 1
0 0 1
-------------------------
-> [SENSORS]: Phát hiện bụi tại (1, 0). Đang hút...
BƯỚC 5:
1 1 1
R 1 1
0 0 1
-------------------------
-> [MODEL]: Các ô liền kề chưa đến: ['LÊN(0, 0)', 'PHẢI(1, 1)']
==> Ưu tiên chọn ô chưa đến: LÊN (0, 0)
BƯỚC 6:
R 1 1
0 1 1
0 0 1
-------------------------
-> [SENSORS]: Phát hiện bụi tại (0, 0). Đang hút...
BƯỚC 7:
R 1 1
0 1 1
0 0 1
-------------------------
-> [MODEL]: Các ô liền kề chưa đến: ['PHẢI(0, 1)']
==> Ưu tiên chọn ô chưa đến: PHẢI (0, 1